# (AT1) Perceptron e KNN em Prática
Trabalho com implementações em NumPy puro.

In [1]:
import numpy as np

def ativacao_degrau(z):
    return 1 if z >= 0 else 0


## Desafio 1 — Classificação Binária com Perceptron Treinável

In [ ]:
X_treino_fraude = np.array([
    [1.5, 1.0],
    [2.0, 2.0],
    [3.5, 1.5],
    [3.0, 3.0],
    [6.5, 5.0],
    [7.0, 7.0],
    [8.5, 6.0],
    [9.0, 8.0]
])

y_treino_fraude = np.array([0, 0, 0, 0, 1, 1, 1, 1])


In [ ]:
def treinar_perceptron(X, y, taxa_aprendizado=0.1, epocas=20):
    pesos = np.ones(X.shape[1], dtype=float)
    bias = 1.0

    for _ in range(epocas):
        houve_erro = False

        for amostra, classe_real in zip(X, y):
            z = np.dot(amostra, pesos) + bias
            classe_predita = ativacao_degrau(z)
            erro = classe_real - classe_predita

            if erro != 0:
                pesos += taxa_aprendizado * erro * amostra
                bias += taxa_aprendizado * erro
                houve_erro = True

        if not houve_erro:
            break

    return pesos, bias


def prever_perceptron(amostra, pesos, bias):
    z = np.dot(amostra, pesos) + bias
    return ativacao_degrau(z)

In [ ]:
taxa_aprendizado = 0.1
epocas = 20

pesos_fraude, bias_fraude = treinar_perceptron(
    X_treino_fraude,
    y_treino_fraude,
    taxa_aprendizado,
    epocas
)

transacao_A = np.array([5.0, 4.0])
transacao_B = np.array([7.0, 6.5])

pred_A = prever_perceptron(transacao_A, pesos_fraude, bias_fraude)
pred_B = prever_perceptron(transacao_B, pesos_fraude, bias_fraude)

print(f"Pesos finais: {pesos_fraude}")
print(f"Bias final: {bias_fraude:.2f}")
print(f"Transação A [5.0, 4.0]: {'Suspeita (Risco de Fraude)' if pred_A == 1 else 'Legítima'}")
print(f"Transação B [7.0, 6.5]: {'Suspeita (Risco de Fraude)' if pred_B == 1 else 'Legítima'}")

## Desafio 2 — Predição de Risco de Churn com KNN

In [2]:
# Desafio 2 
X_treino_churn = np.array([
    [2.0, 1.0],
    [3.0, 0.0],
    [5.0, 1.0],
    [6.0, 2.0],
    [15.0, 4.0],
    [18.0, 5.0],
    [20.0, 4.0],
    [22.0, 6.0]
])

y_treino_churn = np.array([0, 0, 0, 0, 1, 1, 1, 1])

In [ ]:
def classificar_knn(X, y, ponto, k=3, metrica="euclidiana"):
    diferencas = X - ponto

    if metrica.lower() == "euclidiana":
        distancias = np.sqrt(np.sum(diferencas ** 2, axis=1))
    elif metrica.lower() == "manhattan":
        distancias = np.sum(np.abs(diferencas), axis=1)
    else:
        raise ValueError("Métrica deve ser 'euclidiana' ou 'manhattan'.")

    indices_vizinhos = np.argsort(distancias)[:k]
    classes_vizinhas = y[indices_vizinhos]
    classes, contagens = np.unique(classes_vizinhas, return_counts=True)
    classe_predita = classes[np.argmax(contagens)]

    return classe_predita, indices_vizinhos, distancias[indices_vizinhos]

In [ ]:
def exibir_diagnostico_knn(nome, ponto, k=3, metrica="euclidiana"):
    classe, indices, distancias = classificar_knn(
        X_treino_churn, y_treino_churn, ponto, k, metrica
    )

    texto_classe = "Alto Risco" if classe == 1 else "Baixo Risco"

    print(f"{nome}: {ponto}")
    print(f"Classe predita: {texto_classe}")
    print(f"Índices dos vizinhos: {indices}")
    print(f"Distâncias: {np.round(distancias, 4)}")
    print()


cliente_1 = np.array([13.0, 10.0])
cliente_2 = np.array([7.0, 1.5])

exibir_diagnostico_knn("Cliente 1", cliente_1, k=3)
exibir_diagnostico_knn("Cliente 2", cliente_2, k=5)

# Desafio 3 — Recomendação de Servidores Cloud por Similaridade Espacial

In [ ]:


catalogo_servidores = np.array([
    [2.0, 4.0, 50.0],
    [4.0, 8.0, 100.0],
    [8.0, 16.0, 250.0],
    [16.0, 32.0, 500.0],
    [32.0, 64.0, 1000.0],
    [64.0, 128.0, 2000.0]
])

nomes_servidores = [
    "Micro Instância Web",
    "Standard App Server",
    "Medium Backend & Cache",
    "Database Enterprise",
    "High Performance Computing",
    "Big Data & AI Training"
]

perfil_demandado = np.array([12.0, 28.0, 850.0])

In [ ]:
def recomendar_servidores(catalogo, perfil, k=2):
    diferencas = catalogo - perfil
    distancias = np.sqrt(np.sum(diferencas ** 2, axis=1))
    indices_ordenados = np.argsort(distancias)
    indices_recomendados = indices_ordenados[:k]

    return indices_recomendados, distancias[indices_recomendados]

In [ ]:
indices, distancias = recomendar_servidores(
    catalogo_servidores,
    perfil_demandado,
    k=2
)

print(f"Perfil demandado: vCPUs={perfil_demandado[0]:.0f}, RAM={perfil_demandado[1]:.0f} GB, SSD={perfil_demandado[2]:.0f} GB")
print("\nRanking de recomendações:\n")

for posicao, (indice, distancia) in enumerate(zip(indices, distancias), start=1):
    servidor = catalogo_servidores[indice]
    print(f"{posicao}º lugar — {nomes_servidores[indice]}")
    print(f"   Especificações: {servidor[0]:.0f} vCPUs | {servidor[1]:.0f} GB RAM | {servidor[2]:.0f} GB SSD")
    print(f"   Distância: {distancia:.4f}\n")